In [1]:
%%writefile xtts_worker.py

from pathlib import Path
from typing import Optional

import torch
from TTS.api import TTS


# XTTS v2 model
DEFAULT_MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"


def load_xtts_model(
    model_name: str = DEFAULT_MODEL_NAME,
    use_gpu: bool = True,
) -> TTS:
    """
    Load the XTTS-v2 model once.

    Parameters
    ----------
    model_name : str
        XTTS model name.

    use_gpu : bool
        Use CUDA when available.

    Returns
    -------
    TTS
        Loaded XTTS model.
    """

    device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"

    print(f"Loading XTTS model...")
    print(f"Device: {device}")

    tts = TTS(
        model_name=model_name,
        progress_bar=True,
    ).to(device)

    print("XTTS model loaded successfully.")

    return tts


def generate_speech_segment(
    tts: TTS,
    text: str,
    speaker_wav: str,
    output_path: str,
    language: str,
) -> str:
    """
    Convert one text segment into speech using XTTS-v2.

    Parameters
    ----------
    tts : TTS
        Already-loaded XTTS model.

    text : str
        Text to convert into speech.

    speaker_wav : str
        Reference voice WAV file.

    output_path : str
        Destination WAV file.

    language : str
        XTTS language code, e.g. "en", "hi", "fr".

    Returns
    -------
    str
        Path to generated WAV file.
    """

    if not text or not text.strip():
        raise ValueError("Text cannot be empty.")

    speaker_path = Path(speaker_wav)

    if not speaker_path.exists():
        raise FileNotFoundError(
            f"Speaker WAV file does not exist: {speaker_path}"
        )

    output_file = Path(output_path)

    # Make sure output directory exists
    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    print()
    print("----------------------------------------")
    print("Generating XTTS speech")
    print("----------------------------------------")
    print(f"Text      : {text}")
    print(f"Speaker   : {speaker_path}")
    print(f"Language  : {language}")
    print(f"Output    : {output_file}")

    tts.tts_to_file(
        text=text,
        speaker_wav=str(speaker_path),
        language=language,
        file_path=str(output_file),
    )

    print(f"Generated : {output_file}")

    return str(output_file)


def generate_segments(
    tts: TTS,
    segments: list,
    speaker_wav: str,
    output_dir: str = "files/segments",
    language: str = "en",
) -> list:
    """
    Generate WAV files for multiple text segments.

    Each segment should contain a text field.

    Example:
        segments = [
            {"text": "Hello everyone."},
            {"text": "Welcome to the video."},
        ]

    Parameters
    ----------
    tts : TTS
        Already-loaded XTTS model.

    segments : list
        List of dictionaries containing segment text.

    speaker_wav : str
        Reference voice WAV.

    output_dir : str
        Directory where WAV files are saved.

    language : str
        XTTS language code.

    Returns
    -------
    list
        List of generated audio paths.
    """

    output_directory = Path(output_dir)
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    generated_files = []

    for index, segment in enumerate(segments, start=1):

        text = segment["text"]

        output_path = (
            output_directory
            / f"segment_{index:03d}.wav"
        )

        generated_file = generate_speech_segment(
            tts=tts,
            text=text,
            speaker_wav=speaker_wav,
            output_path=str(output_path),
            language=language,
        )

        generated_files.append(generated_file)

    return generated_files

Overwriting xtts_worker.py


In [2]:
from xtts_worker import (
    load_xtts_model,
    generate_speech_segment,
    generate_segments
)

print("XTTS worker imported successfully!")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


XTTS worker imported successfully!


In [6]:

import os

print("Current working directory:", os.getcwd())
print("\nFiles and folders here:")
print(os.listdir("."))

Current working directory: /Users/chaitanyabhardwaj

Files and folders here:
['clustered_data.csv', 'ffmpeg-bin', 'hello.dSYM', 'Music', 'Data Visualization.ipynb', 'First Program.ipynb', 'Decorators.ipynb', 'Simple Linear Regression.ipynb', 'newProject', 'Function.ipynb', 'Untitled1.ipynb', 'hello.c', 'seaborn.ipynb', 'xtts_worker.py', '.DS_Store', 'ANN.ipynb', 'nltk_data', 'hello_world', '.vscode-shared', 'xtts_git.ipynb', '.CFUserTextEncoding', 'HelloWorld.class', 'Logistic Regression.ipynb', 'marks.csv', 'Untitled3.ipynb', 'Untitled.ipynb', 'scikit learn install.ipynb', 'firstprogram.ipynb', 'Iteration and Generator.ipynb', '.local', 'List.ipynb', 'pandas library.ipynb', 'Modules.ipynb', 'Pictures', 'seabornmethod.ipynb', '.zprofile', 'Data Preprocessing.ipynb', 'intro numpy.ipynb', 'KNN.ipynb', 'WebScrapping.ipynb', '.zsh_history', 'hello_world.c', 'Untitled2.ipynb', 'Dogs and Cats Recognition.ipynb', '.ipython', 'Desktop', 'chaitanya.c', '.hackerearth', 'Library', '.matplotlib', 

In [9]:

from pathlib import Path

matches = list(
    Path.cwd().rglob("test_audio_text_translated.json")
)

if matches:
    for path in matches:
        print("Found:", path.resolve())
else:
    print("JSON file not found in the current project directory.")


Found: /Users/chaitanyabhardwaj/vocult/files/test_audio_text_translated.json


In [10]:
import json

with open(
    "/Users/chaitanyabhardwaj/vocult/files/test_audio_text_translated.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])

{
  "source_language": "eng_Latn",
  "target_language": "hin_Deva",
  "segments": [
    {
      "start": 0.291,
      "end": 1.893,
      "speaker": "SPEAKER_00",
      "original_text": "As humans, we understand each other.",
      "translated_text": "मनुष्य के रूप में, हम एक दूसरे को समझते हैं।"
    },
    {
      "start": 2.273,
      "end": 2.734,
      "speaker": "SPEAKER_00",
      "original_text": "Roger that.",
      "translated_text": "समझ में आ गया."
    },
    {
      "start": 3.174,
      "end": 3.815,
      "speaker": "SPEAKER_01",
      "original_text": "I feel ya.",
      "translated_text": "मैं तुम्हें महसूस करता हूँ."
    },
    {
      "start": 4.215,
      "end": 5.016,
      "speaker": "SPEAKER_01",
      "original_text": "Aye-aye, Captain.",
      "translated_text": "हाँ-हाँ, कप्तान."
    },
    {
      "start": 5.697,
      "end": 6.918,
      "speaker": "SPEAKER_01",
      "original_text": "Wait, what are we talking about?",
      "translated_text": "प्रतीक्षा, हम

In [11]:
from pathlib import Path

print("Current Jupyter directory:")
print(Path.cwd())

Current Jupyter directory:
/Users/chaitanyabhardwaj


In [12]:
%cd /Users/chaitanyabhardwaj/vocult

/Users/chaitanyabhardwaj/vocult


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [13]:
!pwd
!ls

/Users/chaitanyabhardwaj/vocult
audio_extractor_worker.py  README.md
files                      transcribe_audio_worker.py
main.py                    translate_text_worker.py


In [14]:
from pathlib import Path

json_path = Path("files/test_audio_text_translated.json")

print("JSON exists:", json_path.exists())
print("JSON path:", json_path.resolve())

JSON exists: True
JSON path: /Users/chaitanyabhardwaj/vocult/files/test_audio_text_translated.json


In [15]:
!ls files

output                          test_audio.mp3
segments                        test_video.mp4
speaker_reference.wav           test.mp4
test_audio_text_translated.json tts_segments
test_audio_text.json


In [16]:
import json

with open(
    "files/test_audio_text_translated.json",
    "r",
    encoding="utf-8"
) as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])

{
  "source_language": "eng_Latn",
  "target_language": "hin_Deva",
  "segments": [
    {
      "start": 0.291,
      "end": 1.893,
      "speaker": "SPEAKER_00",
      "original_text": "As humans, we understand each other.",
      "translated_text": "मनुष्य के रूप में, हम एक दूसरे को समझते हैं।"
    },
    {
      "start": 2.273,
      "end": 2.734,
      "speaker": "SPEAKER_00",
      "original_text": "Roger that.",
      "translated_text": "समझ में आ गया."
    },
    {
      "start": 3.174,
      "end": 3.815,
      "speaker": "SPEAKER_01",
      "original_text": "I feel ya.",
      "translated_text": "मैं तुम्हें महसूस करता हूँ."
    },
    {
      "start": 4.215,
      "end": 5.016,
      "speaker": "SPEAKER_01",
      "original_text": "Aye-aye, Captain.",
      "translated_text": "हाँ-हाँ, कप्तान."
    },
    {
      "start": 5.697,
      "end": 6.918,
      "speaker": "SPEAKER_01",
      "original_text": "Wait, what are we talking about?",
      "translated_text": "प्रतीक्षा, हम

In [17]:
!git checkout model
!git pull origin model

Already on 'model'
Your branch and 'origin/model' have diverged,
and have 1 and 1 different commits each, respectively.
  (use "git pull" if you want to integrate the remote branch with yours)
From https://github.com/gery420/vocult
 * branch            model      -> FETCH_HEAD
hint: You have divergent branches and need to specify how to reconcile them.
hint: You can do so by running one of the following commands sometime before
hint: your next pull:
hint:
hint:   git config pull.rebase false  # merge
hint:   git config pull.rebase true   # rebase
hint:   git config pull.ff only       # fast-forward only
hint:
hint: You can replace "git config" with "git config --global" to set a default
hint: preference for all repositories. You can also pass --rebase, --no-rebase,
hint: or --ff-only on the command line to override the configured default per
hint: invocation.
fatal: Need to specify how to reconcile divergent branches.


In [18]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.13.0
CUDA available: False


In [19]:
from pathlib import Path

BASE_DIR = Path.cwd()

TRANSLATED_JSON = BASE_DIR / "files" / "test_audio_text_translated.json"
SEGMENTS_DIR = BASE_DIR / "files" / "segments"

print("Repository:", BASE_DIR)
print("Translated JSON:", TRANSLATED_JSON)
print("JSON exists:", TRANSLATED_JSON.exists())

SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

print("Segments directory:", SEGMENTS_DIR)

Repository: /Users/chaitanyabhardwaj/vocult
Translated JSON: /Users/chaitanyabhardwaj/vocult/files/test_audio_text_translated.json
JSON exists: True
Segments directory: /Users/chaitanyabhardwaj/vocult/files/segments


In [20]:
%pip install pydub


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
import imageio_ffmpeg

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

print("FFmpeg path:")
print(ffmpeg_path)

FFmpeg path:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1


In [23]:
input_audio = Path("files/test_audio.mp3")

In [24]:
%cd /Users/chaitanyabhardwaj/vocult

from pathlib import Path

print("Current directory:")
print(Path.cwd())

print("\nAudio exists:")
print(Path("files/test_audio.mp3").exists())

print("\nTranslated JSON exists:")
print(Path("files/test_audio_text_translated.json").exists())

/Users/chaitanyabhardwaj/vocult
Current directory:
/Users/chaitanyabhardwaj/vocult

Audio exists:
True

Translated JSON exists:
True


In [25]:
from pathlib import Path

segments_dir = Path("files/segments")
tts_dir = Path("files/tts_segments")
output_dir = Path("files/output")

segments_dir.mkdir(parents=True, exist_ok=True)
tts_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print("Folders ready.")

Folders ready.


In [26]:
import subprocess
from pathlib import Path
import imageio_ffmpeg

input_audio = Path("files/test_audio.mp3")
speaker_wav = Path("files/speaker_reference.wav")

print("Input:", input_audio)
print("Input exists:", input_audio.exists())

if not input_audio.exists():
    raise FileNotFoundError(f"Audio not found: {input_audio}")

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

command = [
    ffmpeg,
    "-y",
    "-i", str(input_audio),
    "-t", "10",
    "-ac", "1",
    "-ar", "22050",
    "-c:a", "pcm_s16le",
    str(speaker_wav)
]

result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("FFmpeg failed")

print("✅ Created:", speaker_wav)
print("Exists:", speaker_wav.exists())

Input: files/test_audio.mp3
Input exists: True
✅ Created: files/speaker_reference.wav
Exists: True


In [27]:
import json
from pathlib import Path

json_path = Path("files/test_audio_text_translated.json")

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Type:", type(data))

if isinstance(data, dict):
    print("Keys:", data.keys())

    for key, value in data.items():
        print(
            f"{key}: type={type(value)}, "
            f"length={len(value) if hasattr(value, '__len__') else 'N/A'}"
        )

Type: <class 'dict'>
Keys: dict_keys(['source_language', 'target_language', 'segments', 'metadata'])
source_language: type=<class 'str'>, length=8
target_language: type=<class 'str'>, length=8
segments: type=<class 'list'>, length=42
metadata: type=<class 'dict'>, length=3


In [28]:
segments = data["segments"]

print("Total segments:", len(segments))

print("\nFirst segment:")
print(json.dumps(segments[0], indent=2, ensure_ascii=False))

Total segments: 42

First segment:
{
  "start": 0.291,
  "end": 1.893,
  "speaker": "SPEAKER_00",
  "original_text": "As humans, we understand each other.",
  "translated_text": "मनुष्य के रूप में, हम एक दूसरे को समझते हैं।"
}


In [29]:
for i, segment in enumerate(segments):
    print(f"\nSegment {i}")
    print("Start:", segment["start"])
    print("End:", segment["end"])
    print("Speaker:", segment["speaker"])
    print("Text:", segment["translated_text"])


Segment 0
Start: 0.291
End: 1.893
Speaker: SPEAKER_00
Text: मनुष्य के रूप में, हम एक दूसरे को समझते हैं।

Segment 1
Start: 2.273
End: 2.734
Speaker: SPEAKER_00
Text: समझ में आ गया.

Segment 2
Start: 3.174
End: 3.815
Speaker: SPEAKER_01
Text: मैं तुम्हें महसूस करता हूँ.

Segment 3
Start: 4.215
End: 5.016
Speaker: SPEAKER_01
Text: हाँ-हाँ, कप्तान.

Segment 4
Start: 5.697
End: 6.918
Speaker: SPEAKER_01
Text: प्रतीक्षा, हम क्या बात कर रहे हैं?

Segment 5
Start: 7.419
End: 8.5
Speaker: SPEAKER_00
Text: खैर, ज्यादातर समय.

Segment 6
Start: 8.86
End: 11.984
Speaker: SPEAKER_00
Text: यहां तक कि अगर हम कुछ शब्दों को स्पष्ट रूप से नहीं सुनते हैं, तो हम इसका सार समझ सकते हैं।

Segment 7
Start: 12.684
End: 14.607
Speaker: SPEAKER_00
Text: अब, मशीनें भी हमें समझ सकती हैं।

Segment 8
Start: 15.047
End: 21.033
Speaker: SPEAKER_00
Text: हमारी घड़ियाँ, कारें, फोन, ड्राइव-थ्रू सभी हमारे शब्दों को संसाधित कर सकते हैं और एक इंसान की तरह प्रतिक्रिया कर सकते हैं।

Segment 9
Start: 21.734
End: 23.295
Speake

In [30]:
from pathlib import Path

speaker_wav = Path("files/speaker_reference.wav")

print("Speaker reference exists:", speaker_wav.exists())
print("Path:", speaker_wav)

Speaker reference exists: True
Path: files/speaker_reference.wav


In [31]:
from TTS.api import TTS

print("TTS imported successfully")

TTS imported successfully


In [32]:
from TTS.api import TTS

tts = TTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2"
)

print("XTTS loaded successfully")

XTTS loaded successfully


In [33]:
import imageio_ffmpeg

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

print(ffmpeg_path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1


In [34]:
import subprocess

result = subprocess.run(
    [ffmpeg_path, "-version"],
    capture_output=True,
    text=True
)

print(result.stdout.splitlines()[0])
print("Return code:", result.returncode)

ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers
Return code: 0


In [35]:
from pathlib import Path

speaker_wav = Path("files/speaker_reference.wav")

print("Exists:", speaker_wav.exists())
print("Path:", speaker_wav.resolve())
print("Size:", speaker_wav.stat().st_size if speaker_wav.exists() else "N/A")

Exists: True
Path: /Users/chaitanyabhardwaj/vocult/files/speaker_reference.wav
Size: 441132


In [45]:
import soundfile as sf

audio_path = "files/speaker_reference.wav"

waveform, sample_rate = sf.read(audio_path)

print("Waveform shape:", waveform.shape)
print("Sample rate:", sample_rate)
print("Duration:", len(waveform) / sample_rate)

Waveform shape: (220500,)
Sample rate: 22050
Duration: 10.0


In [44]:

import sys
import torch
import torchaudio
import torchcodec
import soundfile as sf

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)
print("TorchCodec:", getattr(torchcodec, "__version__", "unknown"))
print("CUDA available:", torch.cuda.is_available())

audio_path = "files/speaker_reference.wav"

# Confirm soundfile can decode the reference WAV.
audio, sample_rate = sf.read(audio_path)
print("SoundFile:", audio.shape, sample_rate)

# Test the exact decoder used by TorchAudio.
try:
    waveform, sr = torchaudio.load(audio_path)
    print("TorchAudio loaded:", waveform.shape, sr)
except Exception as exc:
    print("TorchAudio error:", repr(exc))


Python: 3.13.0 (v3.13.0:60403a5409f, Oct  7 2024, 00:37:40) [Clang 15.0.0 (clang-1500.3.9.4)]
PyTorch: 2.13.0
TorchAudio: 2.11.0
TorchCodec: 0.16.0
CUDA available: False
SoundFile: (220500,) 22050
TorchAudio error: RuntimeError('Failed to create AudioDecoder for files/speaker_reference.wav: Could not load libtorchcodec. Likely causes:\n          1. FFmpeg is not properly installed in your environment. We support\n             versions 4, 5, 6, 7, 8, and 9, and we attempt to load libtorchcodec\n             for each of those versions. Errors for versions not installed on\n             your system are expected; only the error for your installed FFmpeg\n             version is relevant. On Windows, ensure you\'ve installed the\n             "full-shared" version which ships DLLs.\n          2. The PyTorch version (2.13.0) is not compatible with\n             this version of TorchCodec. Refer to the version compatibility\n             table:\n             https://github.com/pytorch/torchco

In [1]:

import subprocess
import torch
import torchaudio

print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)

try:
    import torchcodec
    print("TorchCodec:", torchcodec.__version__)
except Exception as e:
    print("TorchCodec import error:", repr(e))

try:
    print(subprocess.run(
        ["ffmpeg", "-version"],
        capture_output=True,
        text=True,
        check=True
    ).stdout.splitlines()[0])
except Exception as e:
    print("FFmpeg error:", repr(e))


PyTorch: 2.13.0
TorchAudio: 2.11.0
TorchCodec: 0.16.0
FFmpeg error: FileNotFoundError(2, 'No such file or directory')


In [5]:
import os
import shutil

os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

print("FFmpeg path:", shutil.which("ffmpeg"))

FFmpeg path: /opt/homebrew/bin/ffmpeg


In [7]:
import subprocess

result = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout.splitlines()[0])

ffmpeg version 9.0.2 Copyright (c) 2000-2026 the FFmpeg developers


In [8]:
import os
import subprocess
import torch
import torchaudio

print("FFmpeg:", subprocess.run(
    ["which", "ffmpeg"], capture_output=True, text=True
).stdout.strip())

print("FFmpeg libraries:")
print(subprocess.run(
    ["brew", "--prefix", "ffmpeg"], capture_output=True, text=True
).stdout.strip())

print("Python:", os.sys.executable)
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)

FFmpeg: /opt/homebrew/bin/ffmpeg
FFmpeg libraries:
/opt/homebrew/opt/ffmpeg
Python: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3
PyTorch: 2.13.0
TorchAudio: 2.11.0


In [14]:
import soundfile as sf
import torch
import torchaudio

def soundfile_load(filepath, *args, **kwargs):
    audio, sample_rate = sf.read(
        str(filepath),
        dtype="float32",
        always_2d=True
    )

    waveform = torch.from_numpy(audio.T.copy())

    frame_offset = kwargs.get("frame_offset", 0)
    num_frames = kwargs.get("num_frames", -1)

    if frame_offset:
        waveform = waveform[:, frame_offset:]

    if num_frames is not None and num_frames >= 0:
        waveform = waveform[:, :num_frames]

    return waveform, sample_rate

torchaudio.load = soundfile_load

waveform, sample_rate = torchaudio.load(
    "/Users/chaitanyabhardwaj/vocult/files/speaker_reference.wav"
)

print("Waveform shape:", waveform.shape)
print("Sample rate:", sample_rate)

Waveform shape: torch.Size([1, 220500])
Sample rate: 22050


In [15]:
import soundfile as sf

audio, sr = sf.read(
    "/Users/chaitanyabhardwaj/vocult/files/speaker_reference.wav",
    dtype="float32",
    always_2d=True
)

print("Audio shape:", audio.shape)
print("Sample rate:", sr)

Audio shape: (220500, 1)
Sample rate: 22050


In [16]:
import imageio_ffmpeg

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
print(ffmpeg_path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1


In [17]:
import os

os.environ["PATH"] = (
    os.path.dirname(ffmpeg_path)
    + os.pathsep
    + os.environ["PATH"]
)

print("PATH updated")

PATH updated


In [18]:
import os

ffmpeg_dir = os.path.dirname(ffmpeg_path)

os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ["PATH"]

print("Added to PATH:")
print(ffmpeg_dir)

Added to PATH:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries


In [19]:
import imageio_ffmpeg
import os
import subprocess

# Get the FFmpeg executable bundled with imageio-ffmpeg
ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()

print("FFmpeg executable:")
print(ffmpeg_path)

print("\nExists:", os.path.exists(ffmpeg_path))
print("Executable:", os.access(ffmpeg_path, os.X_OK))

# Test it directly
result = subprocess.run(
    [ffmpeg_path, "-version"],
    capture_output=True,
    text=True
)

print("\nFFmpeg version:")
print(result.stdout.splitlines()[0])

FFmpeg executable:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1

Exists: True
Executable: True

FFmpeg version:
ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers


In [20]:
import imageio_ffmpeg
import os
import shutil

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()
ffmpeg_dir = os.path.dirname(ffmpeg_path)

print("FFmpeg path:")
print(ffmpeg_path)

print("\nFFmpeg directory:")
print(ffmpeg_dir)

# Add directory to PATH
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

print("\nPATH updated.")
print("which ffmpeg:", shutil.which("ffmpeg"))

FFmpeg path:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1

FFmpeg directory:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries

PATH updated.
which ffmpeg: /opt/homebrew/bin/ffmpeg


In [21]:
import os
import imageio_ffmpeg
from pathlib import Path

ffmpeg_path = Path(imageio_ffmpeg.get_ffmpeg_exe())

bin_dir = Path.home() / "ffmpeg-bin"
bin_dir.mkdir(exist_ok=True)

link_path = bin_dir / "ffmpeg"

if link_path.exists() or link_path.is_symlink():
    link_path.unlink()

link_path.symlink_to(ffmpeg_path)

os.environ["PATH"] = str(bin_dir) + os.pathsep + os.environ.get("PATH", "")

print("FFmpeg link:", link_path)
print("Target:", ffmpeg_path)

FFmpeg link: /Users/chaitanyabhardwaj/ffmpeg-bin/ffmpeg
Target: /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1


In [22]:
import shutil
import subprocess

print("Detected:", shutil.which("ffmpeg"))

result = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True
)

print(result.stdout.splitlines()[0])

Detected: /Users/chaitanyabhardwaj/ffmpeg-bin/ffmpeg
ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers


In [51]:
import imageio_ffmpeg
from pathlib import Path

ffmpeg_path = Path(imageio_ffmpeg.get_ffmpeg_exe())
ffmpeg_dir = ffmpeg_path.parent

print("FFmpeg:", ffmpeg_path)
print("\nFiles in FFmpeg directory:")

for f in ffmpeg_dir.iterdir():
    print(f.name)

FFmpeg: /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1

Files in FFmpeg directory:
__init__.py
__pycache__
README.md
ffmpeg-macos-aarch64-v7.1


In [23]:
import os
import imageio_ffmpeg
from pathlib import Path

# Actual FFmpeg binary
ffmpeg_path = Path(imageio_ffmpeg.get_ffmpeg_exe())

# Create a personal bin directory
bin_dir = Path.home() / "ffmpeg-bin"
bin_dir.mkdir(exist_ok=True)

# Create a normal "ffmpeg" name
ffmpeg_link = bin_dir / "ffmpeg"

# Remove old link/file if it exists
if ffmpeg_link.exists() or ffmpeg_link.is_symlink():
    ffmpeg_link.unlink()

# Create symbolic link
ffmpeg_link.symlink_to(ffmpeg_path)

# Add it to PATH for this Jupyter kernel
os.environ["PATH"] = str(bin_dir) + os.pathsep + os.environ.get("PATH", "")

print("Actual FFmpeg:")
print(ffmpeg_path)

print("\nFFmpeg command:")
print(ffmpeg_link)

Actual FFmpeg:
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/imageio_ffmpeg/binaries/ffmpeg-macos-aarch64-v7.1

FFmpeg command:
/Users/chaitanyabhardwaj/ffmpeg-bin/ffmpeg


In [24]:
import shutil
import subprocess

print("Detected:", shutil.which("ffmpeg"))

result = subprocess.run(
    ["ffmpeg", "-version"],
    capture_output=True,
    text=True
)

print("\nVersion:")
print(result.stdout.splitlines()[0])

Detected: /Users/chaitanyabhardwaj/ffmpeg-bin/ffmpeg

Version:
ffmpeg version 7.1 Copyright (c) 2000-2024 the FFmpeg developers


In [28]:
from pathlib import Path
import torch
from TTS.api import TTS

# Correct project directory
PROJECT_DIR = Path.home() / "vocult"
SPEAKER_WAV = PROJECT_DIR / "files" / "speaker_reference.wav"
OUTPUT_WAV = PROJECT_DIR / "files" / "xtts_test.wav"

TEXT = "Hello, welcome to Vocult. This is a test of our AI voice translation system."
LANGUAGE = "en"

if not SPEAKER_WAV.is_file():
    raise FileNotFoundError(f"File not found: {SPEAKER_WAV}")

OUTPUT_WAV.parent.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading XTTS on:", device)

tts = TTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2",
    progress_bar=True
).to(device)

tts.tts_to_file(
    text=TEXT,
    speaker_wav=str(SPEAKER_WAV),
    language=LANGUAGE,
    file_path=str(OUTPUT_WAV)
)

print("Generated:", OUTPUT_WAV.resolve())

Loading XTTS on: cpu
Generated: /Users/chaitanyabhardwaj/vocult/files/xtts_test.wav


In [30]:
from pathlib import Path

SEGMENTS = [
    {"id": 1, "text": "Hello, welcome to Vocult."},
    {"id": 2, "text": "This is our AI-powered dubbing system."},
    {"id": 3, "text": "We can translate speech into another language."}
]

SEGMENTS_DIR = Path("files/segments")
SEGMENTS_DIR.mkdir(parents=True, exist_ok=True)

generated_files = []

for segment in SEGMENTS:
    segment_id = segment["id"]
    text = segment["text"].strip()

    if not text:
        continue

    output_file = SEGMENTS_DIR / f"segment_{segment_id:03d}.wav"

    tts.tts_to_file(
        text=text,
        speaker_wav=str(SPEAKER_WAV),
        language="en",
        file_path=str(output_file)
    )

    generated_files.append(str(output_file))
    print("Generated:", output_file)

print("Total segments generated:", len(generated_files))

Generated: files/segments/segment_001.wav
Generated: files/segments/segment_002.wav
Generated: files/segments/segment_003.wav
Total segments generated: 3


In [31]:

import json
from pathlib import Path
from TTS.api import TTS
import torch
import soundfile as sf

# Paths
PROJECT = Path.home() / "vocult"
JSON_PATH = PROJECT / "files" / "test_audio_text_translated.json"
SPEAKER_WAV = PROJECT / "files" / "speaker_reference.wav"
OUTPUT_DIR = PROJECT / "files" / "segments"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load translated Hindi text
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

segments = data["segments"]

# Load XTTS model
device = "cuda" if torch.cuda.is_available() else "cpu"

tts = TTS(
    model_name="tts_models/multilingual/multi-dataset/xtts_v2"
).to(device)

# Generate Hindi speech
for i, segment in enumerate(segments, start=1):
    text = segment.get("translated_text", "").strip()

    if not text:
        continue

    output_file = OUTPUT_DIR / f"segment_{i:03d}.wav"

    tts.tts_to_file(
        text=text,
        speaker_wav=str(SPEAKER_WAV),
        language="hi",
        file_path=str(output_file)
    )

    print(f"Generated: {output_file}")

print("Hindi speech generation complete!")


Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_001.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_002.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_003.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_004.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_005.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_006.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_007.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_008.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_009.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_010.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_011.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_012.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_013.wav
Generated: /Users/chaitanyabhardwaj/vo

The text length exceeds the character limit of 150 for language 'hi', this might cause truncated audio: इसके बजाय हम उसे अनुभव देते हैं, हम उसे ऑडियो और ट...


Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_026.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_027.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_028.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_029.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_030.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_031.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_032.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_033.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_034.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_035.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_036.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_037.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_038.wav
Generated: /Users/chaitanyabhardwaj/vo

The text length exceeds the character limit of 150 for language 'hi', this might cause truncated audio: तो अगली बार जब आप खुद को एक सुखद आवाज का अनुभव करत...


Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_041.wav
Generated: /Users/chaitanyabhardwaj/vocult/files/segments/segment_042.wav
Hindi speech generation complete!


In [37]:
from IPython.display import Audio, display
from pathlib import Path

audio_path = Path(
    "/Users/chaitanyabhardwaj/vocult/files/segments/segment_033.wav"
)

if audio_path.exists():
    display(Audio(filename=str(audio_path)))
else:
    print("Audio file not found:", audio_path)

In [38]:
from IPython.display import Audio, display
from pathlib import Path

audio_path = Path(
    "/Users/chaitanyabhardwaj/vocult/files/segments/segment_027.wav"
)

if audio_path.exists():
    display(Audio(filename=str(audio_path)))
else:
    print("Audio file not found:", audio_path)

In [39]:
from IPython.display import Audio, display
from pathlib import Path

audio_path = Path(
    "/Users/chaitanyabhardwaj/vocult/files/segments/segment_026.wav"
)

if audio_path.exists():
    display(Audio(filename=str(audio_path)))
else:
    print("Audio file not found:", audio_path)